# Objective 2.1 — Predicting Match Goal Difference

## Goal

Build and evaluate a linear regression model that predicts the **goal
difference** (`team1_goals - team2_goals`) for each of the **104 matches**
of the 2026 FIFA World Cup, using **8 explanatory variables that are all
knowable before kickoff**.

| # | Variable | Description |
|---|----------|--------------|
| 1 | `rank_diff` | FIFA ranking position difference (team1 − team2; lower rank number = stronger team) |
| 2 | `age_diff` | Squad average-age difference |
| 3 | `value_diff` | Squad market-value difference (EUR m, Transfermarkt) |
| 4 | `titles_diff` | Prior World Cup titles difference (before 2026) |
| 5 | `host_diff` | Host-nation indicator difference (∈ {−1, 0, 1}) |
| 6 | `rest_diff` | Days-of-rest-before-match difference |
| 7 | `same_confed` | 1 if both teams share a confederation, else 0 |
| 8 | `knockout` | 1 if the match is in the knockout stage, else 0 |

None of these variables depend on anything that happens *during* the match
itself (no shots, possession, cards, etc.), satisfying the "available
before the match" requirement.

In [ ]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from data_prep import load_matches, load_teams, build_match_level_dataset

pd.set_option("display.max_columns", None)
np.random.seed(42)

matches = load_matches()
teams = load_teams()
reg_data = build_match_level_dataset(matches, teams)
print(reg_data.shape)
reg_data.head()

## 1. Sanity checks & exploratory data analysis

In [ ]:
assert reg_data.shape[0] == 104, f"Expected 104 rows, got {reg_data.shape[0]}"
feature_cols = [
    "rank_diff", "age_diff", "value_diff", "titles_diff",
    "host_diff", "rest_diff", "same_confed", "knockout",
]
assert len(feature_cols) == 8
print(reg_data[feature_cols + ["goal_diff"]].isna().sum())
reg_data[feature_cols + ["goal_diff"]].describe()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(reg_data["goal_diff"], bins=range(int(reg_data["goal_diff"].min()) - 1, int(reg_data["goal_diff"].max()) + 2), edgecolor="white")
ax.set_title("Distribution of match goal difference (team1 - team2), n=104")
ax.set_xlabel("Goal difference")
plt.tight_layout()
plt.savefig("../report/figs/reg1_target_hist.png", dpi=120)
plt.show()

corr = reg_data[feature_cols + ["goal_diff"]].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
plt.title("Correlation matrix")
plt.tight_layout()
plt.savefig("../report/figs/reg1_corr.png", dpi=120)
plt.show()
corr["goal_diff"].sort_values(ascending=False)

## 2. Multicollinearity check (VIF)

Variance Inflation Factors above ~5–10 would indicate problematic
redundancy among explanatory variables that could inflate coefficient
standard errors.

In [ ]:
X_all = sm.add_constant(reg_data[feature_cols])
vif = pd.DataFrame(
    {
        "feature": X_all.columns,
        "VIF": [variance_inflation_factor(X_all.values, i) for i in range(X_all.shape[1])],
    }
)
vif

## 3. Train/test split

We hold out 20% of matches (≈21 matches) as a test set, purely for
out-of-sample evaluation of predictive performance; the full training set
is then used to fit and interpret the OLS model with `statsmodels`
(coefficients, standard errors, and p-values).

In [ ]:
X = reg_data[feature_cols]
y = reg_data["goal_diff"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print("Train:", X_train.shape, " Test:", X_test.shape)

## 4. Fit and interpret the OLS model (training set)

In [ ]:
X_train_sm = sm.add_constant(X_train)
ols_model = sm.OLS(y_train, X_train_sm).fit()
print(ols_model.summary())

## 5. Residual diagnostics

In [ ]:
fitted = ols_model.fittedvalues
resid = ols_model.resid

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].scatter(fitted, resid, alpha=0.7)
ax[0].axhline(0, color="red", linestyle="--")
ax[0].set_xlabel("Fitted values")
ax[0].set_ylabel("Residuals")
ax[0].set_title("Residuals vs. fitted")

sm.qqplot(resid, line="45", fit=True, ax=ax[1])
ax[1].set_title("Q-Q plot of residuals")

ax[2].hist(resid, bins=15, edgecolor="white")
ax[2].set_title("Residual distribution")
plt.tight_layout()
plt.savefig("../report/figs/reg1_diagnostics.png", dpi=120)
plt.show()

from scipy import stats as sstats
shapiro_stat, shapiro_p = sstats.shapiro(resid)
print(f"Shapiro-Wilk normality test on residuals: stat={shapiro_stat:.4f}, p={shapiro_p:.4f}")

## 6. Out-of-sample evaluation (held-out test matches)

In [ ]:
X_test_sm = sm.add_constant(X_test, has_constant="add")[X_train_sm.columns]
y_pred = ols_model.predict(X_test_sm)

rmse = mean_squared_error(y_test, y_pred) ** 0.5
mae = mean_absolute_error(y_test, y_pred)
r2_test = r2_score(y_test, y_pred)

n_train, k = X_train.shape[0], X_train.shape[1]
adj_r2_train = 1 - (1 - ols_model.rsquared) * (n_train - 1) / (n_train - k - 1)

print(f"Training R-squared:          {ols_model.rsquared:.4f}")
print(f"Training adjusted R-squared: {adj_r2_train:.4f}")
print(f"Test R-squared:              {r2_test:.4f}")
print(f"Test RMSE:                   {rmse:.3f} goals")
print(f"Test MAE:                    {mae:.3f} goals")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred, alpha=0.75)
lims = [min(y_test.min(), y_pred.min()) - 1, max(y_test.max(), y_pred.max()) + 1]
ax.plot(lims, lims, "r--")
ax.set_xlabel("Actual goal difference")
ax.set_ylabel("Predicted goal difference")
ax.set_title("Test set: predicted vs. actual")
plt.tight_layout()
plt.savefig("../report/figs/reg1_pred_vs_actual.png", dpi=120)
plt.show()

## 7. Conclusion

*(Auto-filled after running the cells above with real data. Summarize:
which of the 8 explanatory variables are statistically significant
predictors of goal difference (p < 0.05) and the sign/interpretation of
their coefficients, the model's overall fit (R², adjusted R²) and
out-of-sample RMSE/MAE, and any diagnostic concerns such as residual
non-normality or heteroscedasticity.)*